## 1.    Setup

# LPG Cylinder Detector — YOLOv11x v2

## Overview
Trains the detection stage of the two-stage pipeline: given a raw scene photo, locate and
draw a bounding box around the LPG cylinder so it can be cropped and handed to the brand
classifiers. This notebook fine-tunes a YOLOv11x model (pretrained on COCO, `yolo11x.pt`)
on a Roboflow-exported cylinder-detection dataset, then evaluates it (standard + TTA
validation, PR/F1 curves) and saves the resulting weights as the "v2" detector — the
predecessor of the currently-shipped `models/yolov11x_lpg_v1_best.pt`, per this notebook's
own file-naming (see Current Status).

## How to Run
1. **Runtime:** Colab GPU — this run used an **A100-SXM4-40GB** (~22 min training time for
   50 epochs at `imgsz=640`, `batch=8`). A T4 will work but will be substantially slower;
   CPU-only is not practical for YOLO training at this scale.
2. **Upload/mount:** Cell 1 mounts Google Drive (for saving results) and then prompts an
   interactive upload of the Roboflow dataset zip (`files.upload()` — this run used
   `"LPG Identification v2.v3i.yolov11.zip"`).
3. **Execution order:** Run top to bottom. Section 3 runs `model.train(...)` (the long step);
   section 4 runs both standard and TTA `model.val(...)`, then downloads/loads the resulting
   chart PNGs. Section 5 copies the trained `best.pt` to a renamed file. Cell 6 (markdown) is
   a plain note recording where `best.pt` landed after training — not something to run.
4. **Expected outputs:** trained weights (`best.pt`, then copied to
   `yolov11x_lpg_v1_best.pt`), YOLO's own run-directory artifacts (curves, confusion
   matrices, `results.png`) under the Drive `project`/`name` directory, and inline
   evaluation summaries — see the Output Files section near the end of the notebook.

## Model / Dataset Info
| | |
|---|---|
| Architecture | YOLOv11x (`yolo11x.pt` pretrained start, `freeze=20`, 56.9M params per section-3 header) |
| Dataset | Roboflow export `LPG Identification v2.v3i.yolov11.zip` — train 1070 / valid 287 / test 70 images, `data.yaml` |
| Classes | Single-class cylinder detector (bounding box only; brand is determined downstream by the classifier notebooks) |
| Expected accuracy | mAP50 **0.969**, mAP50-95 0.825, Precision 0.925, Recall 0.924 (standard val); with TTA: mAP50 0.972, mAP50-95 0.839. Optimal confidence threshold from the F1 curve is 0.374, but the summary print recommends **0.45** for production. |

## Current Status
Training and evaluation ran to completion and outputs are still embedded in this notebook
(cell 5's ~50-epoch training log, and cell 9/11's validation + chart-generation output).
The final saved artifact is named `yolov11x_lpg_v1_best.pt` in this notebook's own save
step (section 5) — this matches the filename of the model currently shipped in
`models/yolov11x_lpg_v1_best.pt` and its mAP50 of 0.969 matches the figure recorded in
`CLAUDE.md`/README for the current detector, despite this notebook's filename being
`detection_v2_yolov11x_model.ipynb`. Treat the "v2" in the notebook's filename as a notebook
revision marker, not a model-version claim distinct from the shipped v1 weights.

In [ ]:
!pip install ultralytics -q
from google.colab import files, drive
drive.mount('/content/drive')
import zipfile, os

print("Uploading Roboflow dataset zip")
uploaded = files.upload()

## 2.   Unzip dataset

In [ ]:
import zipfile, os

zip_name = "LPG Identification v2.v3i.yolov11.zip"

with zipfile.ZipFile(f"/content/{zip_name}", "r") as z:
    z.extractall("/content/lpg_detection")

print("Unzipped!")

# Find yaml
yaml_path = None
for root, dirs, files_ in os.walk("/content/lpg_detection"):
    for f in files_:
        if f.endswith(".yaml"):
            yaml_path = os.path.join(root, f)
            print(f"Found yaml: {yaml_path}")

# Check splits
for split in ["train", "valid", "test"]:
    img_path = f"/content/lpg_detection/{split}/images"
    if os.path.exists(img_path):
        print(f"{split}: {len(os.listdir(img_path))} images")

from ultralytics import YOLO

# Start from yolo11x pretrained — ignore uploaded best.pt
model = YOLO("yolo11x.pt")

model.train(
    # Dataset
    data="/content/lpg_detection/data.yaml",

    # Training
    epochs=50,
    imgsz=640,
    batch=8,

    # Optimizer — Adam + cosine
    optimizer="Adam",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=0.0005,

    # Fine-tuning — freeze first 20 layers (backbone) so only the head/neck adapt to this dataset
    freeze=20,
    patience=15,

    # Multi-scale training — vary input resolution across batches for scale robustness
    multi_scale=True,

    # Augmentations
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    erasing=0.4,

    # Class weighted loss
    cls=0.5,
    box=7.5,
    dfl=1.5,

    # Save to Drive
    project="/content/drive/MyDrive/LPG Cylinder Detection and Classification/Detection",  # ← UPDATE THIS PATH
    name="yolov11x_v1",
    exist_ok=True
)

print("Training complete!")

In [ ]:
from ultralytics import YOLO

# Start from yolo11x pretrained — ignore uploaded best.pt
model = YOLO("yolo11x.pt")

model.train(
    # Dataset
    data="/content/lpg_detection/data.yaml",

    # Training
    epochs=50,
    imgsz=640,
    batch=8,

    # Optimizer — Adam + cosine
    optimizer="Adam",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=0.0005,

    # Fine-tuning
    freeze=20,
    patience=15,

    # Multi-scale training
    multi_scale=True,

    # Augmentations
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    erasing=0.4,

    # Class weighted loss
    cls=0.5,
    box=7.5,
    dfl=1.5,

    # Save to Drive
    project="/content/drive/MyDrive/LPG Cylinder Detection and Classification/Detection",
    name="yolov11x_v1",
    exist_ok=True
)

print("Training complete!")

Best weights saved at: /content/drive/MyDrive/LPG Cylinder Detection and Classification/Detection/yolov11x_v1/weights/best.pt

## 4. Model Evaluation:

## 4.1 PR curve + finding optimal threshold

In [ ]:
# Standard validation
metrics = model.val(data="/content/lpg_detection/data.yaml")
print(f"\nmAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

# TTA validation — potential accuracy boost
metrics_tta = model.val(data="/content/lpg_detection/data.yaml", augment=True)
print(f"\nWith TTA:")
print(f"mAP50:     {metrics_tta.box.map50:.3f}")
print(f"mAP50-95:  {metrics_tta.box.map:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

train_dir = "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Detection/yolov11x_v1"  # ← UPDATE THIS PATH

# ── Load all charts ────────────────────────────────────────────────────────
charts = {
    "BoxF1_curve.png":                  "F1-Confidence Curve",
    "BoxPR_curve.png":                  "Precision-Recall Curve",
    "BoxP_curve.png":                   "Precision-Confidence Curve",
    "BoxR_curve.png":                   "Recall-Confidence Curve",
    "confusion_matrix_normalized.png":  "Confusion Matrix (Normalised)",
    "results.png":                      "Training Results",
}

explanations = {
    "BoxF1_curve.png": (
        "F1 is the harmonic mean of Precision and Recall — the best single metric for detection.\n"
        "Peak F1 = 0.92 at confidence = 0.374 → this is your optimal threshold.\n"
        "The flat top (0.15–0.85) means the model is robust to threshold changes."
    ),
    "BoxPR_curve.png": (
        "Shows the tradeoff between Precision and Recall across all thresholds.\n"
        "mAP50 = 0.969 — area under this curve. Closer to top-right = better.\n"
        "Your curve hugs the top-right corner — excellent detection performance."
    ),
    "BoxP_curve.png": (
        "Precision = of all detections made, how many were correct.\n"
        "High precision = fewer false positives (e.g. detecting a bucket as a cylinder).\n"
        "Rises steeply after conf=0.5 — model becomes very selective at higher thresholds."
    ),
    "BoxR_curve.png": (
        "Recall = of all real cylinders, how many were detected.\n"
        "High recall = fewer missed cylinders.\n"
        "Stays high until conf=0.85 — model misses very few cylinders below that threshold."
    ),
    "confusion_matrix_normalized.png": (
        "Shows what the model predicted vs what was actually there.\n"
        "Diagonal = correct predictions. Off-diagonal = errors.\n"
        "Background row = false positives (non-cylinders detected as cylinders)."
    ),
    "results.png": (
        "Full training curves — box loss, cls loss, dfl loss, and validation metrics over 50 epochs.\n"
        "All losses decreasing smoothly = healthy training, no overfitting.\n"
        "mAP50 and mAP50-95 both improving consistently until final epochs."
    ),
}

# ── Display ────────────────────────────────────────────────────────────────
for fname, title in charts.items():
    fpath = f"{train_dir}/{fname}"
    if not os.path.exists(fpath):
        print(f"Not found: {fname}")
        continue

    fig, ax = plt.subplots(figsize=(12, 7))
    img = mpimg.imread(fpath)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.show()

    # Print explanation below each chart
    print(f" {title}")
    print("─" * 60)
    print(explanations[fname])
    print(f"\n{'═'*60}\n")

# ── Summary box ───────────────────────────────────────────────────────────
print("=" * 60)
print("YOLO11x Detection Model — Final Summary")
print("=" * 60)
print(f"  mAP50:               0.969")
print(f"  mAP50-95:            0.826")
print(f"  Precision:           0.925")
print(f"  Recall:              0.924")
print(f"  F1 Score:            0.92")
print(f"  Optimal conf thresh: 0.374  →  use 0.45 in production")
print(f"  TTA mAP50:           0.972  →  enable augment=True at inference")
print(f"  Training time:       22 mins on A100")
print(f"  Model size:          114.4 MB")
print("=" * 60)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

train_dir = "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Detection/yolov11x_v1"

# ── Load all charts ────────────────────────────────────────────────────────
charts = {
    "BoxF1_curve.png":                  "F1-Confidence Curve",
    "BoxPR_curve.png":                  "Precision-Recall Curve",
    "BoxP_curve.png":                   "Precision-Confidence Curve",
    "BoxR_curve.png":                   "Recall-Confidence Curve",
    "confusion_matrix_normalized.png":  "Confusion Matrix (Normalised)",
    "results.png":                      "Training Results",
}

explanations = {
    "BoxF1_curve.png": (
        "F1 is the harmonic mean of Precision and Recall — the best single metric for detection.\n"
        "Peak F1 = 0.92 at confidence = 0.374 → this is your optimal threshold.\n"
        "The flat top (0.15–0.85) means the model is robust to threshold changes."
    ),
    "BoxPR_curve.png": (
        "Shows the tradeoff between Precision and Recall across all thresholds.\n"
        "mAP50 = 0.969 — area under this curve. Closer to top-right = better.\n"
        "Your curve hugs the top-right corner — excellent detection performance."
    ),
    "BoxP_curve.png": (
        "Precision = of all detections made, how many were correct.\n"
        "High precision = fewer false positives (e.g. detecting a bucket as a cylinder).\n"
        "Rises steeply after conf=0.5 — model becomes very selective at higher thresholds."
    ),
    "BoxR_curve.png": (
        "Recall = of all real cylinders, how many were detected.\n"
        "High recall = fewer missed cylinders.\n"
        "Stays high until conf=0.85 — model misses very few cylinders below that threshold."
    ),
    "confusion_matrix_normalized.png": (
        "Shows what the model predicted vs what was actually there.\n"
        "Diagonal = correct predictions. Off-diagonal = errors.\n"
        "Background row = false positives (non-cylinders detected as cylinders)."
    ),
    "results.png": (
        "Full training curves — box loss, cls loss, dfl loss, and validation metrics over 50 epochs.\n"
        "All losses decreasing smoothly = healthy training, no overfitting.\n"
        "mAP50 and mAP50-95 both improving consistently until final epochs."
    ),
}

# ── Display ────────────────────────────────────────────────────────────────
for fname, title in charts.items():
    fpath = f"{train_dir}/{fname}"
    if not os.path.exists(fpath):
        print(f"Not found: {fname}")
        continue

    fig, ax = plt.subplots(figsize=(12, 7))
    img = mpimg.imread(fpath)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.show()

    # Print explanation below each chart
    print(f" {title}")
    print("─" * 60)
    print(explanations[fname])
    print(f"\n{'═'*60}\n")

# ── Summary box ───────────────────────────────────────────────────────────
print("=" * 60)
print("YOLO11x Detection Model — Final Summary")
print("=" * 60)
print(f"  mAP50:               0.969")
print(f"  mAP50-95:            0.826")
print(f"  Precision:           0.925")
print(f"  Recall:              0.924")
print(f"  F1 Score:            0.92")
print(f"  Optimal conf thresh: 0.374  →  use 0.45 in production")
print(f"  TTA mAP50:           0.972  →  enable augment=True at inference")
print(f"  Training time:       22 mins on A100")
print(f"  Model size:          114.4 MB")
print("=" * 60)

import shutil

src  = "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Detection/yolov11x_v1/weights/best.pt"  # ← UPDATE THIS PATH
dest = "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Detection/yolov11x_v1/weights/yolov11x_lpg_v1_best.pt"  # ← UPDATE THIS PATH

shutil.copy(src, dest)
print("Saved as yolov11x_lpg_v1_best.pt")

In [ ]:
import shutil

src  = "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Detection/yolov11x_v1/weights/best.pt"
dest = "/content/drive/MyDrive/LPG Cylinder Detection and Classification/Detection/yolov11x_v1/weights/yolov11x_lpg_v1_best.pt"

shutil.copy(src, dest)
print("Saved as yolov11x_lpg_v1_best.pt")

## Output Files

| File | Saved to | Contents / purpose |
|---|---|---|
| `best.pt` / `last.pt` | Drive `Detection/yolov11x_v1/weights/` (written by Ultralytics' own `project`/`name` training args) | Ultralytics' standard best/last checkpoint from the `model.train(...)` run. |
| `yolov11x_lpg_v1_best.pt` | Drive `Detection/yolov11x_v1/weights/` (copy of `best.pt`), then downloaded locally via `files.download(dest)` | Renamed copy of the best checkpoint — this is the file consumed as the detector by `src/predict.py` / `src/predict_ensemble.py`. |
| `BoxF1_curve.png`, `BoxPR_curve.png`, `BoxP_curve.png`, `BoxR_curve.png`, `confusion_matrix.png`, `confusion_matrix_normalized.png`, `results.png` | Downloaded locally during `model.val(...)` (cell 9's PR/threshold section), also present under Drive `Detection/yolov11x_v1/` since Ultralytics writes them alongside the run directory | Standard YOLO validation diagnostics — F1/PR/precision/recall vs. confidence curves, confusion matrices, and full training-loss/metric curves. Re-displayed with commentary in the following cell. |

In [ ]:
from google.colab import files
files.download(dest)